
# RoboterProgrammierung Projektaufgabe — Roundtrip-Path 

## Aufgabe c) Multi-Query-Ansatz mit Visibility PRM

### Endbericht Teil a)

    Der verwendete Roundtrip-Planer (IPVisPRMRound -> VisPRMRound) ist eine Erweiterung des VisibilityPRM. Er nutzt dessen Kernlogik zur Erstellung einer effizienten Roadmap, ändert aber wie diese Roadmap für die Pfadsuche verwendet wird.

1. Basis: Nutzung der Visibility-Roadmap Der Planer erbt vom Standard-VisPRM und nutzt exakt dieselbe Methode (_learnRoadmap), um den Graphen zu erstellen. 

    Während der Standard-VisPRM nach dem Bau der Roadmap typischerweise eine Single-Query (Start → Ziel) ausführt, nutzt der VisPRMRound diesen Graphen als Basis für eine komplexe Multi-Query-Operation.

3. Erweiterung: Multi-Goal-Anbindung & Skalierung
Im Gegensatz zur Standard-Implementierung, die nur Start und Ziel anbindet, muss der Roundtrip-Planer eine ganze Liste von Zielkonfigurationen in den Graphen integrieren.

    Hierfür wird ein k-d-Tree verwendet, um effizient die nächsten Nachbarn im Visibility-Graphen zu finden.

    Besonderheit: Für Planar-Roboter (ab 3 Dimensionen) führt der Planer eine Skalierung der Winkel-Dimension (Theta) durch (SCALE_FACTOR_THETA = 0.05). Diese Anpassung verhindert, dass die Winkeldimension die euklidischen Distanzen im Arbeitsraum dominiert.

4. Abstraktion: TSP-Layer
Der VisPRMRound abstrahiert den Visibility-Graphen zu einem TSP-Graphen. Er berechnet mittels Dijkstra die tatsächlichen Pfadkosten durch die Roadmap zwischen allen Zielpunkten (Distanzmatrix). Darauf basierend wird das Traveling Salesman Problem gelöst (Christofides), um die optimale Reihenfolge zu ermitteln, bevor die Teilstücke wieder zu einem Gesamtpfad zusammengesetzt werden.

In [ ]:
import sys
sys.path.append(".")
sys.path.append("./collisionChecker")

import matplotlib.pyplot as plt
import time
from IPython.display import display, HTML

from IPTestSuiteBenchmark import benchList
from IPVisibilityPRMRound import VisPRMRound
from PathAnimator import PathAnimator

### Testsetup zu veranschaulichung der Funktion von VisPRMRound

In [ ]:
# 1. SETUP
target_bench_name = "spinner_3DoF"
benchmark = next((b for b in benchList if b.name == target_bench_name), None)

if not benchmark:
    print(f"Benchmark '{target_bench_name}' nicht gefunden. Nehme den ersten verfügbaren.")
    benchmark = benchList[0]

print(f"Umgebung geladen: {benchmark.name}")

# 2. PLANUNG
env = benchmark.collisionChecker
planner = VisPRMRound(env)
config = {"ntry": 500}

print("Plane Pfad...")
start_time = time.time()
path_coords = planner.planPath([benchmark.startList[0]], benchmark.goalList, config)
duration = time.time() - start_time

if path_coords and len(path_coords) > 0:
    print(f"Pfad gefunden! Länge: {len(path_coords)} Punkte in {duration:.3f}s")

    # 3. Animation
    print("Starte PathAnimator...")

    is_3d = hasattr(env, 'drawRobot')

    animator = PathAnimator(
        benchmark,
        path_coords,
        "VisPRMRound",
        duration
    )

    animator.animate(
        step_size=2.0 if is_3d else 1.0,
        interval_ms=50,
        show_inline=True,
        max_frames=300
    )

else:
    print("Kein Pfad gefunden. Versuche 'ntry' zu erhöhen.")

Der Planer nutzt zur für die erstellung der Roadmap und die Lösung des Traveling-Sales-Person-Probelms die entsprechende Networkx-Funktion.

Diese wiederum nutzt den Christofides-Alogrihtmus, welcher zusätzlich den Kruskal's Algorithmus (Minimaler Spannbaum) und die Eulersche Tour (Hierholzer) zur Aproximation der Reihenfolge nutzt. Diesen haben wir gewählt, da er den Optimalen Pfad um mindestend den Faktor 1,5 erreicht und somit eine naha Aproximation an das Ergebnis verspricht. Zusätzlich stellt die NetworkX-Funktion ein passendes Gesamtpacket da.

![My figure](./images/christofides.png)

### Endbericht Teil b)

b) Optimierung und Glättung von Bewegungsbahnen

Da probabilistische Planer wie der PRM (Probabilistic Roadmap) oder RRT (Rapidly-exploring Random Trees) auf zufälligen Stichproben basieren, sind die initial generierten Pfade oft suboptimal. Sie zeichnen sich durch unnötige Zick-Zack-Bewegungen und scharfe Kanten an den Wegpunkten aus. Dies ist mechanisch ungünstig, da der Roboter an jedem Eckpunkt abbremsen (Geschwindigkeit = 0) und neu beschleunigen müsste.

Eine etablierte Vorgehensweise zur Optimierung ist die Shortcut-Heuristik (Path Pruning).
Funktionsweise der Shortcut-Methode

Diese Methode arbeitet als "Post-Processing"-Schritt auf dem bereits gefundenen, kollisionsfreien Pfad. Der Algorithmus versucht, den Pfad iterativ abzukürzen, indem er prüft, ob Zwischenpunkte übersprungen werden können.

Weitere Möglichkeiten (Trajektorienglättung)

Während die Shortcut-Methode den Pfad geometrisch verkürzt, bleiben die Übergänge an den verbleibenden Wegpunkten oft noch eckig. Um eine wirkliche Glättung für eine flüssige Roboterbewegung zu erreichen, können im Anschluss Spline-Verfahren (z. B. B-Splines oder Bezier-Kurven) angewendet werden. Dabei werden die Wegpunkte der Shortcut-Methode als Stützstellen verwendet, durch die eine differenzierbare Kurve gelegt wird, was eine Bewegung ohne Stopps ermöglicht.